# Multimodal World Model Analysis

This notebook demonstrates how to analyze multimodal world models, probe different sensory channels, and auto-discover learned concepts across modalities.

## Overview
- Setup multimodal model environment
- Probe individual sensory channels
- Auto-discover cross-modal concepts
- Analyze modality-specific and shared representations

In [ ]:
# Setup and imports
import torch
import numpy as np
from world_model_lens.analysis.multimodal import probe_multimodal, auto_discover_concepts
from world_model_lens import HookedWorldModel
from world_model_lens.utils.multimodal_dataset import MultimodalDataset
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)

print("Multimodal analysis environment setup complete")

## Setup Multimodal Model

Load a multimodal world model that processes vision, audio, and possibly other sensory inputs.

In [ ]:
# Load multimodal world model
model = HookedWorldModel.from_pretrained("multimodal-dreamerv3")

# List available modalities
modalities = model.get_modalities()
print(f"Available modalities: {modalities}")

# Check model architecture
print(f"\nModel architecture summary:")
print(f"  Vision encoder: {model.vision_dim}D")
if hasattr(model, "audio_encoder"):
    print(f"  Audio encoder: {model.audio_dim}D")
print(f"  Shared representation: {model.representation_dim}D")
print(f"  Transition model hidden: {model.hidden_dim}D")

## Create Multimodal Dataset

Generate or load a dataset with synchronized multimodal observations.

In [ ]:
# Create synthetic multimodal dataset
dataset = MultimodalDataset(
    num_samples=5000,
    sequence_length=16,
    modalities=["vision", "audio"],
    vision_dim=12288,   # 64x64x3 RGB
    audio_dim=1024,     # Spectrogram features
    action_dim=18,
    include_labels=True
)

print(f"Dataset created with {len(dataset)} samples")
print(f"Sequence length: {dataset.sequence_length}")
print(f"Modalities: {dataset.modalities}")

# Sample batch
sample = dataset[0]
print(f"\nSample shapes:")
for modality, data in sample["observations"].items():
    print(f"  {modality}: {data.shape}")

## Probe Sensory Channels

Use linear probing and other techniques to understand what each sensory channel learns.

In [ ]:
# Define probe configurations for each modality
from world_model_lens.analysis.multimodal import ProbeConfig

probe_configs = {
    "vision": ProbeConfig(
        layer_name="encoder.vision",
        probe_type="linear",
        target_labels=["object_presence", "spatial_relation", "color"],
        train_fraction=0.7
    ),
    "audio": ProbeConfig(
        layer_name="encoder.audio",
        probe_type="linear",
        target_labels=["sound_type", "intensity", "pitch"],
        train_fraction=0.7
    ),
    "shared": ProbeConfig(
        layer_name="encoder.shared",
        probe_type="mlp",
        target_labels=["object_presence", "sound_type", "reward"],
        train_fraction=0.7
    )
}

In [ ]:
# Run probes on each modality
probe_results = {}

for modality, config in probe_configs.items():
    print(f"\nProbing {modality} channel...")
    
    results = probe_multimodal(
        model=model,
        dataset=dataset,
        probe_config=config,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    probe_results[modality] = results
    
    print(f"  Accuracy: {results['accuracy']:.4f}")
    print(f"  F1 Score: {results['f1_score']:.4f}")

In [ ]:
# Visualize probe performance comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
ax1 = axes[0]
modalities = list(probe_results.keys())
accuracies = [probe_results[m]["accuracy"] for m in modalities]
colors = ["#3498db", "#e74c3c", "#2ecc71"]
ax1.bar(modalities, accuracies, color=colors)
ax1.set_xlabel("Modality/Representation")
ax1.set_ylabel("Probe Accuracy")
ax1.set_title("Probe Accuracy by Channel")
ax1.set_ylim([0, 1])
ax1.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)

# Per-label performance heatmap
ax2 = axes[1]
label_matrix = []
all_labels = set()
for modality, results in probe_results.items():
    for label, metrics in results["per_label"].items():
        all_labels.add(label)

for modality in modalities:
    row = []
    for label in sorted(all_labels):
        row.append(probe_results[modality]["per_label"].get(label, {}).get("accuracy", 0))
    label_matrix.append(row)

sns.heatmap(
    label_matrix,
    xticklabels=sorted(all_labels),
    yticklabels=modalities,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    ax=ax2
)
ax2.set_title("Per-Label Probe Accuracy")
ax2.set_xlabel("Target Label")
ax2.set_ylabel("Channel")

plt.tight_layout()
plt.savefig("probe_results.png", dpi=150)
plt.show()

## Auto-Discover Concepts

Automatically discover concepts learned by the model without requiring labels.

In [ ]:
# Auto-discover concepts across modalities
discover_config = {
    "min_concept_size": 50,
    "max_concepts": 100,
    "cross_modality_threshold": 0.7,
    "clustering_method": "hierarchical",
    "distance_metric": "cosine"
}

print("Auto-discovering concepts...")
concepts = auto_discover_concepts(
    model=model,
    dataset=dataset,
    config=discover_config
)

print(f"\nDiscovered {len(concepts)} concepts")

In [ ]:
# Analyze discovered concepts
print("Concept Summary:")
print("="*60)

for i, concept in enumerate(sorted(concepts, key=lambda x: -x["importance"])[:10]):
    print(f"\nConcept {i+1}: {concept['name']}")
    print(f"  Importance: {concept['importance']:.4f}")
    print(f"  Modality: {concept['primary_modality']}")
    print(f"  Activation rate: {concept['activation_rate']:.2%}")
    print(f"  Cluster size: {concept['cluster_size']}")
    if concept.get("cross_modal"):
        print(f"  Cross-modal links: {concept['cross_modal_links']}")

In [ ]:
# Visualize concept space
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Concept importance distribution
ax1 = axes[0]
importances = [c["importance"] for c in concepts]
ax1.hist(importances, bins=30, color="#9b59b6", edgecolor="white")
ax1.set_xlabel("Concept Importance")
ax1.set_ylabel("Count")
ax1.set_title("Distribution of Concept Importance Scores")
ax1.axvline(x=np.median(importances), color="red", linestyle="--", label=f"Median: {np.median(importances):.3f}")
ax1.legend()

# Modality distribution
ax2 = axes[1]
modality_counts = {}
for c in concepts:
    mod = c["primary_modality"]
    modality_counts[mod] = modality_counts.get(mod, 0) + 1

ax2.pie(
    modality_counts.values(),
    labels=modality_counts.keys(),
    autopct="%1.1f%%",
    colors=["#3498db", "#e74c3c", "#2ecc71", "#f39c12"],
    explode=[0.05] * len(modality_counts)
)
ax2.set_title("Concept Distribution by Modality")

plt.tight_layout()
plt.savefig("concept_discovery.png", dpi=150)
plt.show()

## Analyze Cross-Modal Associations

Investigate how concepts learned in different modalities relate to each other.

In [ ]:
# Find cross-modal concept associations
cross_modal_concepts = [c for c in concepts if c.get("cross_modal", False)]

print(f"Found {len(cross_modal_concepts)} cross-modal concepts")
print("\nTop Cross-Modal Associations:")
print("-"*60)

for concept in sorted(cross_modal_concepts, key=lambda x: -x["cross_modal_strength"])[:5]:
    print(f"  {concept['name']}")
    print(f"    Vision concepts: {concept.get('vision_concepts', [])}")
    print(f"    Audio concepts: {concept.get('audio_concepts', [])}")
    print(f"    Association strength: {concept['cross_modal_strength']:.4f}")
    print()

In [ ]:
# Create concept association heatmap
vision_concepts = [c for c in concepts if c["primary_modality"] == "vision"][:10]
audio_concepts = [c for c in concepts if c["primary_modality"] == "audio"][:10]

# Compute association matrix
association_matrix = np.zeros((len(vision_concepts), len(audio_concepts)))
for i, v_concept in enumerate(vision_concepts):
    for j, a_concept in enumerate(audio_concepts):
        association_matrix[i, j] = v_concept.get(a_concept["name"], 0)

plt.figure(figsize=(10, 8))
sns.heatmap(
    association_matrix,
    xticklabels=[c["name"][:15] for c in audio_concepts],
    yticklabels=[c["name"][:15] for c in vision_concepts],
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f"
)
plt.xlabel("Audio Concepts")
plt.ylabel("Vision Concepts")
plt.title("Cross-Modal Concept Association Matrix")
plt.tight_layout()
plt.savefig("cross_modal_associations.png", dpi=150)
plt.show()

## Save Results

Export all analysis results for later use.

In [ ]:
import json

# Save probe results
probe_json = {
    modality: {
        "accuracy": float(results["accuracy"]),
        "f1_score": float(results["f1_score"]),
        "per_label": {k: {"accuracy": float(v["accuracy"])} for k, v in results["per_label"].items()}
    }
    for modality, results in probe_results.items()
}

# Save concept discovery
concepts_json = [
    {
        "name": c["name"],
        "importance": float(c["importance"]),
        "primary_modality": c["primary_modality"],
        "activation_rate": float(c["activation_rate"]),
        "cross_modal": c.get("cross_modal", False)
    }
    for c in concepts
]

with open("multimodal_probe_results.json", "w") as f:
    json.dump(probe_json, f, indent=2)

with open("discovered_concepts.json", "w") as f:
    json.dump(concepts_json, f, indent=2)

print("Results saved successfully!")
print("  - multimodal_probe_results.json")
print("  - discovered_concepts.json")